In [1]:
import pickle
import pandas as pd
from keras.models import load_model
from keras import Model
from typing import cast
import numpy as np
import streamlit as st
from sklearn.preprocessing import LabelEncoder , OneHotEncoder , StandardScaler

In [30]:
input = pd.read_csv('./training_data/Churn_Modelling.csv')
print(input.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0         90437.74       1  
1         81341.31       0  
2        112912.73       1  
3         95174.52       0  
4        1149

In [31]:
input = input.drop(['RowNumber','CustomerId','Surname','Exited','Tenure','NumOfProducts','IsActiveMember','HasCrCard'],axis=1)
print(input.head())

   CreditScore Geography  Gender  Age    Balance  EstimatedSalary
0          619    France  Female   42       0.00         90437.74
1          608     Spain  Female   41   83807.86         81341.31
2          502    France  Female   42  159660.80        112912.73
3          699    France  Female   39       0.00         95174.52
4          850     Spain  Female   43  125510.82        114968.34


In [32]:
# gender label encoder
gender_label_encoder = LabelEncoder()
input['Gender'] = np.asarray(gender_label_encoder.fit_transform(input['Gender']))

In [33]:
print(input.head())

   CreditScore Geography  Gender  Age    Balance  EstimatedSalary
0          619    France       0   42       0.00         90437.74
1          608     Spain       0   41   83807.86         81341.31
2          502    France       0   42  159660.80        112912.73
3          699    France       0   39       0.00         95174.52
4          850     Spain       0   43  125510.82        114968.34


In [35]:
# One hot encoder for Geography column
geo_onehot_encoder = OneHotEncoder(sparse_output=False)
geo_encoder = np.asarray(geo_onehot_encoder.fit_transform(input[['Geography']]))

print(geo_encoder)

geo_encoder_column_lists = geo_onehot_encoder.get_feature_names_out(['Geography'])

print(geo_encoder_column_lists)

[[1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 ...
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]]
['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [36]:
# create a Geography Data frame
geo_encoder_df = pd.DataFrame(geo_encoder,columns=geo_encoder_column_lists)

print(geo_encoder_df)

      Geography_France  Geography_Germany  Geography_Spain
0                  1.0                0.0              0.0
1                  0.0                0.0              1.0
2                  1.0                0.0              0.0
3                  1.0                0.0              0.0
4                  0.0                0.0              1.0
...                ...                ...              ...
9995               1.0                0.0              0.0
9996               1.0                0.0              0.0
9997               1.0                0.0              0.0
9998               0.0                1.0              0.0
9999               1.0                0.0              0.0

[10000 rows x 3 columns]


In [37]:
# concat input dataframe with Geography Data frame

new_input_df = pd.concat([input.drop(['Geography'],axis=1),geo_encoder_df],axis=1)

print(new_input_df)

      CreditScore  Gender  Age    Balance  EstimatedSalary  Geography_France  \
0             619       0   42       0.00         90437.74               1.0   
1             608       0   41   83807.86         81341.31               0.0   
2             502       0   42  159660.80        112912.73               1.0   
3             699       0   39       0.00         95174.52               1.0   
4             850       0   43  125510.82        114968.34               0.0   
...           ...     ...  ...        ...              ...               ...   
9995          771       1   39       0.00        121250.85               1.0   
9996          516       1   35   57369.61         90850.00               1.0   
9997          709       0   36       0.00         97541.99               1.0   
9998          772       1   42   75075.31        139119.25               0.0   
9999          792       0   28  130142.79        103098.62               1.0   

      Geography_Germany  Geography_Spai

In [39]:
# Split the data into features and target

X = new_input_df.drop(['EstimatedSalary'],axis=1)
Y = new_input_df['EstimatedSalary']

from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [40]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train)


[[ 0.35649971  0.91324755 -0.6557859  ...  1.00150113 -0.57946723
  -0.57638802]
 [-0.20389777  0.91324755  0.29493847 ... -0.99850112  1.72572313
  -0.57638802]
 [-0.96147213  0.91324755 -1.41636539 ... -0.99850112 -0.57946723
   1.73494238]
 ...
 [ 0.86500853 -1.09499335 -0.08535128 ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.15932282  0.91324755  0.3900109  ...  1.00150113 -0.57946723
  -0.57638802]
 [ 0.47065475  0.91324755  1.15059039 ... -0.99850112  1.72572313
  -0.57638802]]


In [41]:
# save all the label encoder , one hot encoder , scaler in a file

import os

os.makedirs('./pkl/regression',exist_ok=True)

with open('./pkl/regression/label_encoder_gender.pkl','wb') as file:
    pickle.dump(gender_label_encoder,file=file)


with open('./pkl/regression/onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(geo_onehot_encoder,file=file)

with open('./pkl/regression/scaler.pkl','wb') as file:
    pickle.dump(scaler,file=file)

#### ANN Regression Problem statement


In [42]:
from keras import Sequential , Input
from keras.layers import Dense
from keras.optimizers import Adam
from keras.losses import MeanAbsoluteError

model = Sequential([
    Input(shape=(X_test.shape[1],)),
    Dense(64,activation="relu"),
    Dense(32,activation="relu"),
    Dense(1)
])

print(model.summary())

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 64)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,625 (10.25 KB)

 Trainable params: 2,625 (10.25 KB)

 Non-trainable params: 0 (0.00 B)

None


In [44]:
## compile the model
opt = Adam(learning_rate=0.01)
loss = MeanAbsoluteError()

model.compile(optimizer=opt,loss=loss,metrics=['mae','mape'])

In [47]:
# TensorBoard , Early Stopping
from keras.callbacks import TensorBoard , EarlyStopping
import datetime

log_dir = "logs/regression/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

earlyStopping_callback = EarlyStopping(monitor="val_loss",patience=20,restore_best_weights=True)


In [48]:
### Train the model
history = model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,callbacks=[tensorboard_callback,earlyStopping_callback])

print(history)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 554us/step - loss: 8634.3857 - mae: 8634.3857 - mape: 9.0234 - val_loss: 8419.3154 - val_mae: 8419.3154 - val_mape: 8.9662
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 490us/step - loss: 8655.3701 - mae: 8655.3701 - mape: 9.0327 - val_loss: 8376.9502 - val_mae: 8376.9502 - val_mape: 8.8840
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 488us/step - loss: 8654.5684 - mae: 8654.5684 - mape: 9.0362 - val_loss: 8357.4434 - val_mae: 8357.4434 - val_mape: 8.8959
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 502us/step - loss: 8636.1982 - mae: 8636.1982 - mape: 9.0319 - val_loss: 8381.1016 - val_mae: 8381.1016 - val_mape: 8.8903
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 482us/step - loss: 8646.5361 - mae: 8646.5361 - mape: 9.0446 - val_loss: 8415.2871 - val_mae: 8415.2871 - val_mape: 8.8619
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 475us/step - loss: 8635.6143 - mae: 8635.6143 - mape: 9.0273 - val_loss: 8395.3213 - val_mae: 8395.3213 - val_mape: 8.848

In [49]:
model.save('./model/salaryRegressionModel.keras')
